In [0]:
import os
import mlflow.spark
from pyspark.sql.functions import col

mlflow_temp_path = "/Volumes/workspace/aml_fraud_project/raw_data/mlflow_tmp"
dbutils.fs.mkdirs(mlflow_temp_path)
os.environ["MLFLOW_DFS_TMP"] = mlflow_temp_path

silver_table_name = "workspace.aml_fraud_project.transactions_silver"
df_ml = spark.table(silver_table_name)


model_name = "workspace.aml_fraud_project.fraud_rf_model"
model_uri = f"models:/{model_name}/1"  

loaded_model = mlflow.spark.load_model(model_uri)

final_predictions = loaded_model.transform(df_ml)

gold_df = final_predictions.filter(col("prediction") == 1.0) \
    .select(
        col("step").alias("Time_Step"),
        col("type").alias("Transaction_Type"),
        col("amount").alias("Transaction_Amount"),
        col("transaction_count_so_far").alias("Velocity_Score"),
        col("errorBalanceOrig").alias("Origin_Balance_Anomaly")
    )

gold_table_name = "workspace.aml_fraud_project.flagged_frauds_gold"
print(f"Saving high-risk transactions to Gold Table: {gold_table_name}")

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(gold_table_name)

print("Gold Layer Complete!")
display(spark.table(gold_table_name).orderBy(col("Transaction_Amount").desc()).limit(10))